In [10]:
import pandas as pd
import json
import time
from pathlib import Path
from pdf2image import convert_from_path
import mimetypes

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials
from google.genai import errors as genai_errors


# ==========================
# CONFIGURATION (COMPANY GATEWAY)
# ==========================
base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ5MjI4MjksImlhdCI6MTc2NDkyMTAzMCwiYXV0aF90aW1lIjoxNzY0OTIxMDI5LCJqdGkiOiI0YzFkMzEwOS1mZDY1LTRkMzAtYmQzZi00OTkzMGU4ZjgyMGYiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6Ijg4YmJiY2EyLWUwNTItNDRkMy04OGRiLTE2OTljOWFlMjk1MiIsImF0X2hhc2giOiJ4YTBjODlIMlh0Z0VUbU9QWnk0N21BIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiI4OGJiYmNhMi1lMDUyLTQ0ZDMtODhkYi0xNjk5YzlhZTI5NTIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.0yjANjzAAIcC0eEZ0ozjliRBa40olYnxesz_XFq81f07nYZ37U5o6Ime84Bn4YeuODAV88mMvy1VNfT62IrGsq7AYNBLpfbXyQQ9B-Slb6w8txqXjt5Hf9DMjlUNugQfQmqUQnDSAQ7hVvjqGgNY08TWTSp_eX26Uou-wthihBmJ1aBRwG5-v9kx4QyjvJ0VNu1O0Vi_YHI4NVRfaLZyRLptqJ7z5WyupoEWocqvG_TP5C3yHCUrtst2oYVVdmqcJ0egotSN8JTcvFp2H0KogpIz-3Q6jXZ4bYLqCHY5oYgt-sRKHxnK2IeQY0zQE8XnkonsDrUIVthj6W_W6d4pAw"
credentials = Credentials(access_token)
client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"

print("Gemini (company gateway) initialized - Connector Table Extract Mode.")


# ==========================
# STRICT CONNECTOR TABLE PROMPT
# ==========================
def get_strict_connector_prompt():
    return """
    You are a PRECISION Connector Table Extraction AI with ZERO TOLERANCE for hallucination.

    GOAL:
    Extract ONLY CONNECTOR TABLES from the drawing, exactly as printed.
    If you are not sure about something, use null or "-" and explain in metadata.

    WHAT TO TARGET:
    - Tables that describe electrical / electronic connectors:
      * Have a connector name (e.g., "Connector X12", "X12", "J101")
      * Optional "Mating device" or similar line
      * Columns like:
        - Pin no. / Pin number / Pin #
        - Signal name / Signal
        - To connector / Destination
        - Pin (destination)
        - Wire gauge / Wire ga. / AWG / Area (mm2)

    WHAT TO IGNORE:
    - BOM / parts lists
    - Legends / symbol keys
    - Notes tables
    - Dimension tables
    - Miscellaneous text blocks

    EXTRACTION RULES (STRICT):
    1) Connector header:
       - connector_name:
         * The connector ID (e.g., "X12", "J101", "C1")
         * Typically appears in or near the table title.
         * If unclear, use null.
       - mating_device:
         * Text after labels like "Mating device:", "Mating connector:"
         * If not present, use null.

    2) Table body:
       - Read header row exactly as printed.
       - Read EVERY data row underneath, in order.
       - Do NOT skip rows with "-" or blanks.
       - Preserve "-" when shown, use null only when cell truly empty.

    3) Row cells:
       Typical columns and meanings (examples – do NOT rename in the table itself):
       - "Pin no." / "Pin number" / "Pin #":
           connector pin number (e.g., "1", "2", "A1")
       - "Signal name" / "Signal":
           net or function (e.g., "M4HRES", "GND_M4H")
       - "To Connector" / "Destination":
           target connector name (e.g., "X06")
       - "Pin":
           destination pin number (e.g., "35")
       - "Wire gauge" / "Wire ga." / "AWG":
           cross-section (e.g., "0.5", "2.5", "20 AWG")

       You MUST:
       - Keep the original header texts as the first row.
       - Keep all rows the same number of columns (pad with null if needed).
       - Use "-" where you see "-" (do not convert "-" to null).

    4) HALLUCINATION GUARD:
       NEVER:
       - Fill in missing pins by pattern.
       - Guess signal names or gauges.
       - Invent new rows.
       - Merge information from other tables.
       - "Fix" data to make it look nicer.

    OUTPUT FORMAT (STRICT JSON):

    {
      "extraction_metadata": {
        "total_tables_found": 1,
        "extraction_confidence": "high",
        "notes": "Short description of any issues or that everything was clear"
      },
      "connector_tables": [
        {
          "connector_name": "X12",
          "mating_device": "IPAM-CA2",
          "row_count": 10,
          "column_count": 5,
          "data": [
            ["Pin no.", "Signal name", "To Connector", "Pin", "Wire gauge"],
            ["1", "M4HRES", "X06", "35", "0.5"],
            ["2", "M4HVAB", "X06", "32", "0.5"],
            ["3", "-", "-", "-", "-"],
            ["4", "M4H1/M4A", "X06", "19", "0.5"],
            ["5", "VXS_M4H", "X06", "34", "0.5"],
            ["6", "GND_M4H", "X06", "58", "0.5"],
            ["7", "M4L1", "X06", "2", "2.5"],
            ["8", "M4L2", "X06", "3", "2.5"],
            ["9", "M4L3", "X06", "1", "2.5"]
          ]
        }
      ]
    }

    If NO connector-like tables are found, return:

    {
      "extraction_metadata": {
        "total_tables_found": 0,
        "extraction_confidence": "high",
        "notes": "No connector tables detected in this sheet"
      },
      "connector_tables": []
    }

    FIELD REQUIREMENTS:
    - extraction_metadata.total_tables_found : integer
    - extraction_metadata.extraction_confidence : "high" | "medium" | "low"
    - extraction_metadata.notes : string

    - connector_tables : list of table objects
      * connector_name : string or null
      * mating_device  : string or null
      * row_count      : integer (header + data rows)
      * column_count   : integer
      * data           : list of rows (list of cells)
        - data[0] is the header row
        - data[1..] are data rows

    VALIDATION CHECKS YOU MUST PASS BEFORE RETURN:
    - All rows in a table have the same number of columns.
    - The row_count and column_count match the data array.
    - You did not create any rows not visible in the drawing.
    - You used "-" exactly where shown and null only for truly empty cells.
    """


# ==========================
# LOW-LEVEL MODEL CALL
# ==========================

def _call_model_on_image_bytes(image_bytes: bytes, mime_type: str) -> dict:
    """Call Gemini on raw image bytes and return parsed JSON dict."""
    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            get_strict_connector_prompt(),
            image_part,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0,   # max determinism
        ),
    )

    raw = response.text or "{}"

    # Strip code fences if any
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0]
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0]

    return json.loads(raw)


# ==========================
# PER-PAGE ANALYSIS
# ==========================
def analyze_connector_page(image_path: str, page_index: int) -> dict:
    """Analyze one page image and return connector_tables + metadata."""
    print(f"   → Page {page_index}: Analyzing {Path(image_path).name} ...")

    try:
        with open(image_path, "rb") as f:
            image_bytes = f.read()
    except Exception as e:
        print(f"      ✗ Failed to read image: {e}")
        return {
            "extraction_metadata": {
                "total_tables_found": 0,
                "extraction_confidence": "low",
                "notes": f"Image read error: {e}",
            },
            "connector_tables": [],
        }

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    try:
        data = _call_model_on_image_bytes(image_bytes, mime_type)
        if not isinstance(data, dict):
            print("      ⚠ Invalid JSON structure from model")
            return {
                "extraction_metadata": {
                    "total_tables_found": 0,
                    "extraction_confidence": "low",
                    "notes": "Model response not a JSON object",
                },
                "connector_tables": [],
            }
        print("      ✓ Model response parsed")
        return data

    except genai_errors.ClientError as e:
        print(f"      ✗ ClientError: {e}")
        return {
            "extraction_metadata": {
                "total_tables_found": 0,
                "extraction_confidence": "low",
                "notes": f"API ClientError: {e}",
            },
            "connector_tables": [],
        }
    except Exception as e:
        print(f"      ✗ Error calling model: {e}")
        return {
            "extraction_metadata": {
                "total_tables_found": 0,
                "extraction_confidence": "low",
                "notes": f"Unexpected error: {e}",
            },
            "connector_tables": [],
        }


# ==========================
# VALIDATION / CLEANUP HELPERS
# ==========================
def _normalize_table(table: dict, table_idx: int) -> dict:
    """
    Validate and normalize a single connector table dict.
    Returns cleaned table dict or None if unusable.
    """
    if not isinstance(table, dict):
        print(f"      ⚠ Table {table_idx}: not a dict")
        return None

    connector_name = table.get("connector_name")
    mating_device = table.get("mating_device")
    data = table.get("data", [])

    if not isinstance(data, list) or len(data) < 2:
        print(f"      ⚠ Table {table_idx}: needs header + at least one data row")
        return None

    # Enforce rectangular table
    n_cols = len(data[0])
    for i, row in enumerate(data):
        if not isinstance(row, list):
            print(f"      ⚠ Table {table_idx} row {i}: not a list")
            return None
        if len(row) != n_cols:
            # pad or trim
            if len(row) < n_cols:
                row.extend([None] * (n_cols - len(row)))
            else:
                del row[n_cols:]

    # Normalize "-" vs empty
    for i in range(len(data)):
        for j in range(len(data[i])):
            cell = data[i][j]
            if isinstance(cell, str):
                s = cell.strip()
                if s in ["-", "–", "—", "−"]:
                    data[i][j] = "-"      # normalized dash
                elif s == "":
                    data[i][j] = None     # true empty
                else:
                    data[i][j] = s        # trimmed string
            elif cell is None:
                data[i][j] = None

    return {
        "connector_name": connector_name,
        "mating_device": mating_device,
        "data": data,
        "row_count": len(data),
        "column_count": n_cols,
    }


# ==========================
# MAIN PROCESSOR
# ==========================
def process_connector_file(file_path: str):
    """
    High-level pipeline (similar style to your dimension extractor):

    - If PDF: convert to images (300 DPI)
    - If image: process directly
    - For each page:
        * call Gemini with strict connector prompt
        * collect connector_tables
    - Write one Excel:
        * Sheet 'Summary'        : one row per connector table
        * Sheet per connector    : its pin table (header + data)
    """
    print("\n" + "="*70)
    print("CONNECTOR TABLE EXTRACTOR - STRICT, NO HALLUCINATION")
    print("="*70)
    print(f"📄 File: {file_path}")

    path = Path(file_path)
    if not path.exists():
        print("✗ ERROR: File not found")
        return

    ext = path.suffix.lower()
    page_images: list[str] = []

    # 1) PDF → images
    if ext == ".pdf":
        print("\n🔄 Converting PDF to images (300 DPI)...")
        try:
            pages = convert_from_path(file_path, dpi=300)
        except Exception as e:
            print(f"✗ ERROR: PDF conversion failed: {e}")
            return

        if not pages:
            print("✗ ERROR: PDF conversion returned 0 pages")
            return

        print(f"   ✓ Converted {len(pages)} page(s)")
        for i, p in enumerate(pages, start=1):
            img_path = f"temp_connector_{i}.png"
            p.save(img_path, "PNG")
            page_images.append(img_path)
    else:
        print("\n   ✓ Processing as single image file")
        page_images = [file_path]

    # 2) Per-page extraction
    all_tables = []
    summary_rows = []

    for page_idx, img in enumerate(page_images, start=1):
        print(f"\n--- Page {page_idx}/{len(page_images)} ---")
        result = analyze_connector_page(img, page_idx)

        meta = result.get("extraction_metadata", {}) or {}
        total_found = meta.get("total_tables_found", 0)
        conf = meta.get("extraction_confidence", "unknown")
        notes = meta.get("notes", "")

        print(f"   → Tables found: {total_found}, confidence: {conf.upper()}")
        if notes:
            print(f"   → Notes: {notes}")

        tables = result.get("connector_tables", []) or []
        if not isinstance(tables, list):
            print("   ⚠ connector_tables is not a list, skipping page")
            continue

        for idx, t in enumerate(tables, start=1):
            normalized = _normalize_table(t, idx)
            if normalized is None:
                continue

            conn_name = normalized["connector_name"] or f"Connector_P{page_idx}_{idx}"
            mating_device = normalized["mating_device"] or None
            row_count = normalized["row_count"]
            col_count = normalized["column_count"]

            print(f"      ✓ Connector: {conn_name}  ({row_count} rows x {col_count} cols)")
            if mating_device:
                print(f"        Mating device: {mating_device}")

            all_tables.append({
                "page": page_idx,
                "connector_name": conn_name,
                "mating_device": mating_device,
                "data": normalized["data"],
                "row_count": row_count,
                "column_count": col_count,
            })

            summary_rows.append({
                "Page": page_idx,
                "Connector_Name": conn_name,
                "Mating_Device": mating_device,
                "Row_Count": row_count,
                "Column_Count": col_count,
                "Page_Confidence": conf,
                "Page_Notes": notes,
            })

    # 3) Excel output
    if not all_tables:
        print("\n⚠ No connector tables extracted. No Excel created.")
        # cleanup temp images if PDF
        if ext == ".pdf":
            for img in page_images:
                Path(img).unlink(missing_ok=True)
        return

    out_name = f"{path.stem}_ConnectorTables.xlsx"
    print("\n" + "-"*70)
    print(f"WRITING EXCEL: {out_name}")
    print("-"*70)

    with pd.ExcelWriter(out_name, engine="openpyxl") as writer:
        # Summary sheet
        df_summary = pd.DataFrame(summary_rows)
        df_summary.to_excel(writer, sheet_name="Summary", index=False)
        print("   ✓ Sheet: Summary")

        used_sheet_names = set(["Summary"])
        for t in all_tables:
            conn_name = t["connector_name"]
            page_idx = t["page"]
            data = t["data"]

            if not data or len(data) < 2:
                continue

            headers = data[0]
            rows = data[1:]
            df = pd.DataFrame(rows, columns=headers)

            # Build sheet name: include page index to avoid duplicates
            base_name = f"P{page_idx}_{conn_name}" if len(page_images) > 1 else str(conn_name)
            # sanitize
            base_name = "".join(ch for ch in base_name if ch not in '\\/*?:[]')
            base_name = base_name[:31]

            sheet_name = base_name
            counter = 1
            while sheet_name in used_sheet_names or not sheet_name:
                suffix = f"_{counter}"
                sheet_name = (base_name[:31-len(suffix)] + suffix) if base_name else f"Sheet_{counter}"
                counter += 1
            used_sheet_names.add(sheet_name)

            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"   ✓ Sheet: {sheet_name} ({len(rows)} rows)")

    print("\n✅ DONE!")
    print(f"   Excel file created: {out_name}")
    print(f"   Connector tables: {len(all_tables)}")
    print(f"   Summary rows    : {len(summary_rows)}")

    # 4) Cleanup temp images
    if ext == ".pdf":
        for img in page_images:
            Path(img).unlink(missing_ok=True)
        print("\n🧹 Cleaned up temporary images.")


Gemini (company gateway) initialized - Connector Table Extract Mode.


In [13]:
process_connector_file("Screenshot 2025-12-05 141750.png")


CONNECTOR TABLE EXTRACTOR - STRICT, NO HALLUCINATION
📄 File: Screenshot 2025-12-05 141750.png

   ✓ Processing as single image file

--- Page 1/1 ---
   → Page 1: Analyzing Screenshot 2025-12-05 141750.png ...
      ✓ Model response parsed
   → Tables found: 16, confidence: HIGH
   → Notes: Successfully extracted 16 connector tables. Two tables (X05 and X06) were physically split into two columns of rows on the drawing; they have been extracted as single, continuous tables as logically intended.
      ✓ Connector: X01  (2 rows x 5 cols)
      ✓ Connector: X02  (2 rows x 5 cols)
      ✓ Connector: X04  (17 rows x 6 cols)
      ✓ Connector: X05  (55 rows x 6 cols)
      ✓ Connector: X06  (68 rows x 6 cols)
      ✓ Connector: X10  (10 rows x 5 cols)
      ✓ Connector: X11  (10 rows x 5 cols)
      ✓ Connector: X12  (10 rows x 5 cols)
      ✓ Connector: X13  (10 rows x 5 cols)
      ✓ Connector: X14  (10 rows x 5 cols)
      ✓ Connector: X16  (3 rows x 5 cols)
      ✓ Connector: X17  (4 r